# 03 — QAOA single run

One QAOA configuration end-to-end on `mag7` with `K=2, p=3`:
the multi-start training curve, the probability histogram, and the top-5
most likely bitstrings (with their true `C(x)`).

Defaults: 10 random `(gamma, beta)` inits, COBYLA `maxiter=200`,
`rhobeg=0.1`. Single-seed QAOA is not meaningful — the landscape is
non-convex enough that the multi-start is mandatory.

In [ ]:
# === Bootstrap (Colab + local) ===
import sys, os
try:
    import google.colab  # noqa: F401
    get_ipython().system('test -d /content/fys5419 || git clone -q https://github.com/egil10/fys5419.git /content/fys5419')
    get_ipython().run_line_magic('cd', '/content/fys5419/project2/code/notebooks')
except ImportError:
    pass
sys.path.append('..')
from scripts.colab import setup; setup()

# === Project imports ===
import numpy as np
import matplotlib.pyplot as plt

from scripts.data      import load_returns
from scripts.baskets   import config
from scripts.portfolio import PortfolioProblem
from scripts.classical import brute_force
from scripts.qaoa      import solve, decode_top_k, make_hamiltonians
from scripts.metrics   import approximation_ratio, prob_optimal, prob_feasible
from scripts.plotting  import apply_style, PALETTE, title, fig_path
apply_style()

In [ ]:
BASKET = 'mag7'
P, K, LAM, AP = 3, 2, 2.0, 0.5

tickers, start, end = config(BASKET)
r  = load_returns(tickers, start, end, cache_name=BASKET)
pf = PortfolioProblem(r.mu, r.Sigma, lam=LAM, A=AP, K=K, tickers=tuple(r.tickers))

bf  = brute_force(pf)
res = solve(pf, p=P, n_restarts=10, seed=42, verbose=False)

print(f'brute force:  x={bf.bitstring}  C={bf.cost:.6f}')
print(f'QAOA p={P}:     E={res["energy"]:.6f}  ratio={res["ratio"]:.4f}')
print(f'P(optimum) = {prob_optimal(res["probs"], bf.x):.4f}')
print(f'P(feasible) = {prob_feasible(res["probs"], pf.n, pf.K):.4f}')

### Top-5 bitstrings

In [ ]:
import pandas as pd
pd.DataFrame(decode_top_k(res['probs'], pf, k=5))[['bitstring', 'prob', 'cost', 'budget']]

### Figures — training restarts + probability histogram

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

# Multi-seed restart energies — shows the landscape is non-convex
energies = [h['fun'] for h in res['history']]
axes[0].bar(range(len(energies)), energies,
            color=PALETTE['blue_muted'], edgecolor=PALETTE['charcoal'])
axes[0].axhline(res['energy'], color=PALETTE['red'], ls='--', lw=1.5,
                label=f'best E = {res["energy"]:.4f}')
axes[0].axhline(bf.cost, color=PALETTE['ochre'], ls=':', lw=1.5,
                label=f'C(x*) = {bf.cost:.4f}')
axes[0].set_xlabel('restart'); axes[0].set_ylabel('converged energy')
title(axes[0], 'Multi-start COBYLA energies',
      '10 random inits; non-convex landscape -> wide spread')
axes[0].legend()

# Probability histogram with budget-feasible bars highlighted
probs = res['probs']
n_states = len(probs)
feas = np.array([bin(k).count('1') == pf.K for k in range(n_states)])
cols = [PALETTE['red'] if f else PALETTE['blue_muted'] for f in feas]
axes[1].bar(range(n_states), probs, color=cols,
            edgecolor=PALETTE['charcoal'], linewidth=0.4)
axes[1].axhline(1.0/n_states, color=PALETTE['grey'], ls=':', lw=1,
                label=f'uniform 1/{n_states}')
axes[1].set_xticks(range(n_states))
axes[1].set_xticklabels([format(k, f'0{pf.n}b') for k in range(n_states)],
                       rotation=90, fontsize=8)
axes[1].set_ylabel('probability')
title(axes[1], 'Measurement probabilities',
      f'red = budget K={pf.K}; optimum is {bf.bitstring}')
axes[1].legend()

plt.tight_layout()
fig.savefig(fig_path('qaoa', f'single_run_{BASKET}_p{P}'), bbox_inches='tight')
plt.show()